# Exploratory notebook: Repository-Context (Multimodal Logistic Regression Classifier)

This notebook is a supplementary analysis for RQ5. It tests whether augmenting
the multimodal logistic regression classifier with six repository-level context
features improves rejection prediction over the baseline 1,221-dimensional
feature vector.

The six context features are computed by `RepositoryContextEncoder` and capture
historical rejection rates at the agent and repository level, fitted on the
training split only to prevent leakage. The augmented model is trained and
evaluated identically to the final multimodal logistic regression classifier to
ensure comparability.

**This notebook is not part of the main model pipeline.** Results are discussed in
the thesis as a supplementary finding.


| Item | Description |
|------|-------|
| Base model | Multimodal logistic regression classifier (1,221-dim) |
| Added features | 6 repository-level context features (RepositoryContextEncoder) |
| Augmented feature vector | 1,227-dim (1,221 base + 6 context) |
| Evaluation metric | PR-AUC (primary), ROC-AUC, Precision, Recall, F1 |
| Comparison | Baseline multimodal LR vs. context-augmented multimodal LR |
| Outputs | `results/exploratory/exploratory_multimodal_lr_repo_context/` |
| Hardware | T4 GPU recommended; requires cached CodeBERT embeddings from `multimodal_lr.ipynb` |

In [1]:
# Colab setup (run first, once per session)
# ---------------------------------------------------------------
# CONFIGURE THIS PATH before running any other cell.
# Set DRIVE_BASE to the folder on your Google Drive that contains
# data/, src/, results/, and requirements.txt for this project.
# Example: if you placed the shared folder at the root of your
# Drive, set this to '/content/drive/MyDrive/AgenticPRRejection'
# ---------------------------------------------------------------

from google.colab import drive
import subprocess, sys
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/Thesis/AgenticPRRejection')   # replace with your own path

# Mount Google Drive
drive.mount('/content/drive')
print('Drive mounted')

# Point directly to src/ on Drive
REPO_SRC = f'{DRIVE_BASE}/src'
sys.path.insert(0, REPO_SRC)
print(f'src/ added to path: {REPO_SRC}')

# Install dependencies once per session
subprocess.run(
    ['pip', 'install', '-q', '-r',
     f'{DRIVE_BASE}/requirements.txt'],
    check=True
)
print('Dependencies installed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted
src/ added to path: /content/drive/MyDrive/Thesis/AgenticPRRejection/src
Dependencies installed


In [2]:
# Imports and seeds
import sys
sys.path.insert(0, DRIVE_BASE / 'src')

import json
import logging
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)
from sklearn.inspection import permutation_importance

from repo_context_features import RepositoryContextEncoder
from features.text_features import PRTextEncoder
from features.metadata_features import PRMetadataEncoder
from features.feature_pipeline import FeaturePipeline
from preprocessing import load_processed_data
from splits import temporal_split

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s — %(message)s',
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Imports and seeds complete.')

Imports and seeds complete.


In [3]:
# Load splits and base data
df = load_processed_data()
train_df, val_df, test_df = temporal_split(df)

print('Split sizes and rejection rates:')
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    rej_rate = split['label'].mean()
    print(f'  {name}: {len(split):,} PRs, rejection rate = {rej_rate:.3f}')

Split sizes and rejection rates:
  Train: 7,453 PRs, rejection rate = 0.383
  Val: 1,598 PRs, rejection rate = 0.330
  Test: 1,597 PRs, rejection rate = 0.352


In [4]:
# Load cached CodeBERT diff embeddings
# Only diff embeddings are loaded from cache
from model1_embedder import load_embeddings

EMBEDDINGS_DIR = Path(DRIVE_BASE) / 'results' / 'final' / 'multimodal_lr' / 'embeddings'

X_diff_train, y_train = load_embeddings('train', EMBEDDINGS_DIR)
X_diff_val,   y_val   = load_embeddings('val',   EMBEDDINGS_DIR)
X_diff_test,  y_test  = load_embeddings('test',  EMBEDDINGS_DIR)

# Load PR IDs to align the DataFrames to the embedding rows.
# The embedding arrays were computed after dropping PRs with no diff chunks,
# so the split DataFrames must be filtered and reordered to match.
train_pr_ids = np.load(EMBEDDINGS_DIR / 'train_pr_ids.npy', allow_pickle=True)
val_pr_ids   = np.load(EMBEDDINGS_DIR / 'val_pr_ids.npy',   allow_pickle=True)
test_pr_ids  = np.load(EMBEDDINGS_DIR / 'test_pr_ids.npy',  allow_pickle=True)

def _align_df(df: pd.DataFrame, pr_ids: np.ndarray) -> pd.DataFrame:
    """Filter and reorder df so its rows match the order of pr_ids."""
    df_indexed = df.set_index('id')
    valid_ids = [pid for pid in pr_ids if pid in df_indexed.index]
    if len(valid_ids) != len(pr_ids):
        n_missing = len(pr_ids) - len(valid_ids)
        print(f'WARNING: {n_missing} pr_ids not found in DataFrame; they will be skipped.')
    return df_indexed.loc[valid_ids].reset_index()

train_df = _align_df(train_df, train_pr_ids)
val_df   = _align_df(val_df,   val_pr_ids)
test_df  = _align_df(test_df,  test_pr_ids)

# Trim embedding arrays to match any dropped rows
train_mask = np.isin(train_pr_ids, train_df['id'].values)
val_mask   = np.isin(val_pr_ids,   val_df['id'].values)
test_mask  = np.isin(test_pr_ids,  test_df['id'].values)
X_diff_train = X_diff_train[train_mask]
y_train      = y_train[train_mask]
X_diff_val   = X_diff_val[val_mask]
y_val        = y_val[val_mask]
X_diff_test  = X_diff_test[test_mask]
y_test       = y_test[test_mask]

assert len(train_df) == len(X_diff_train), (
    f'Train size mismatch: DataFrame {len(train_df)} vs embeddings {len(X_diff_train)}'
)
assert len(val_df)   == len(X_diff_val), (
    f'Val size mismatch: DataFrame {len(val_df)} vs embeddings {len(X_diff_val)}'
)
assert len(test_df)  == len(X_diff_test), (
    f'Test size mismatch: DataFrame {len(test_df)} vs embeddings {len(X_diff_test)}'
)

print('Diff embedding shapes:')
print(f'  X_diff_train: {X_diff_train.shape}')
print(f'  X_diff_val:   {X_diff_val.shape}')
print(f'  X_diff_test:  {X_diff_test.shape}')

Diff embedding shapes:
  X_diff_train: (7453, 768)
  X_diff_val:   (1598, 768)
  X_diff_test:  (1597, 768)


In [5]:
# Fit a fresh FeaturePipeline on training data
# This pipeline is independent of the final Multimodal LR pipeline. It is fit
# from scratch here and saved only to results/exploratory/.
text_encoder     = PRTextEncoder()
metadata_encoder = PRMetadataEncoder()
pipeline = FeaturePipeline(
    active_features=['diff', 'text', 'metadata'],
    text_encoder=text_encoder,
    metadata_encoder=metadata_encoder,
)

pipeline.fit(train_df, X_diff_train=X_diff_train)

X_base_train = pipeline.transform(train_df, X_diff=X_diff_train)
X_base_val   = pipeline.transform(val_df,   X_diff=X_diff_val)
X_base_test  = pipeline.transform(test_df,  X_diff=X_diff_test)

print('Base feature matrix shapes:')
print(f'  X_base_train: {X_base_train.shape}')
print(f'  X_base_val:   {X_base_val.shape}')
print(f'  X_base_test:  {X_base_test.shape}')
print('Group slices:')
for group, slc in pipeline.group_slices_.items():
    print(f'  {group}: columns {slc.start}:{slc.stop} ({slc.stop - slc.start} dims)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Base feature matrix shapes:
  X_base_train: (7453, 1221)
  X_base_val:   (1598, 1221)
  X_base_test:  (1597, 1221)
Group slices:
  diff: columns 0:768 (768 dims)
  text: columns 768:1152 (384 dims)
  metadata: columns 1152:1221 (69 dims)


In [6]:
# Fit RepositoryContextEncoder on training data only
# All rates below are derived solely from train_df.
ctx_enc = RepositoryContextEncoder()
ctx_enc.fit(train_df)

print('Context feature names:', ctx_enc.feature_names_)

X_ctx_train = ctx_enc.transform(train_df)
X_ctx_val   = ctx_enc.transform(val_df)
X_ctx_test  = ctx_enc.transform(test_df)

print('Context feature matrix shapes:')
print(f'  X_ctx_train: {X_ctx_train.shape}')
print(f'  X_ctx_val:   {X_ctx_val.shape}')
print(f'  X_ctx_test:  {X_ctx_test.shape}')

# Descriptive summary of training-set context statistics.
global_rate = train_df['label'].mean()
print(f'\nGlobal training rejection rate: {global_rate:.4f}')

repo_rates_train = X_ctx_train[:, 0]  # repo_rejection_rate (unscaled)
agent_rates_train = X_ctx_train[:, 1]  # agent_rejection_rate (unscaled)

print('Per-repo rejection rate distribution (training rows):')
print(f'  Q0={np.percentile(repo_rates_train, 0):.4f}  '
      f'Q25={np.percentile(repo_rates_train, 25):.4f}  '
      f'Q50={np.percentile(repo_rates_train, 50):.4f}  '
      f'Q75={np.percentile(repo_rates_train, 75):.4f}  '
      f'Q100={np.percentile(repo_rates_train, 100):.4f}')

print('Per-agent rejection rate distribution (training rows):')
print(f'  Q0={np.percentile(agent_rates_train, 0):.4f}  '
      f'Q25={np.percentile(agent_rates_train, 25):.4f}  '
      f'Q50={np.percentile(agent_rates_train, 50):.4f}  '
      f'Q75={np.percentile(agent_rates_train, 75):.4f}  '
      f'Q100={np.percentile(agent_rates_train, 100):.4f}')

Context feature names: ['repo_rejection_rate', 'agent_rejection_rate', 'agent_repo_rejection_rate', 'log1p_repo_pr_count', 'log1p_agent_pr_count', 'pr_size_vs_repo_median']
Context feature matrix shapes:
  X_ctx_train: (7453, 6)
  X_ctx_val:   (1598, 6)
  X_ctx_test:  (1597, 6)

Global training rejection rate: 0.3835
Per-repo rejection rate distribution (training rows):
  Q0=0.0639  Q25=0.2235  Q50=0.3243  Q75=0.5112  Q100=0.9176
Per-agent rejection rate distribution (training rows):
  Q0=0.2717  Q25=0.2717  Q50=0.4156  Q75=0.4843  Q100=0.4843


In [7]:
# Build augmented feature matrices
# Concatenate the base multimodal features with the 6 context features.
# Total dimensionality = base_dim + 6.
X_aug_train = np.concatenate([X_base_train, X_ctx_train.astype(X_base_train.dtype)], axis=1)
X_aug_val   = np.concatenate([X_base_val,   X_ctx_val.astype(X_base_val.dtype)],   axis=1)
X_aug_test  = np.concatenate([X_base_test,  X_ctx_test.astype(X_base_test.dtype)],  axis=1)

print('Augmented feature matrix shapes:')
print(f'  X_aug_train: {X_aug_train.shape}  (base {X_base_train.shape[1]} + context 6)')
print(f'  X_aug_val:   {X_aug_val.shape}')
print(f'  X_aug_test:  {X_aug_test.shape}')

Augmented feature matrix shapes:
  X_aug_train: (7453, 1227)  (base 1221 + context 6)
  X_aug_val:   (1598, 1227)
  X_aug_test:  (1597, 1227)


In [8]:
# Train logistic regression on augmented features
model = LogisticRegression(
    C=0.1,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=SEED,
)
model.fit(X_aug_train, y_train)
print('Logistic regression trained.')

Logistic regression trained.


In [9]:
# Evaluate and compare against baseline Multimodal LR
# Baseline values are hardcoded from the final multimodal_lr.ipynb run.
BASELINE = {
    'val':  {'pr_auc': 0.5374, 'roc_auc': 0.6687, 'precision': 0.458,  'recall': 0.5473, 'f1': 0.4987},
    'test': {'pr_auc': 0.5309, 'roc_auc': 0.6744, 'precision': 0.4992, 'recall': 0.5516, 'f1': 0.5241},
}

def _evaluate(X, y, split_name):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    pr_auc  = average_precision_score(y, probs)
    roc_auc = roc_auc_score(y, probs)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y, preds, average='binary', zero_division=0
    )
    return {'split': split_name, 'pr_auc': pr_auc, 'roc_auc': roc_auc,
            'precision': prec, 'recall': rec, 'f1': f1}

val_metrics  = _evaluate(X_aug_val,  y_val,  'val')
test_metrics = _evaluate(X_aug_test, y_test, 'test')

print('Results comparison (threshold = 0.5):')
print(f'{"Metric":<22} {"Baseline val":>14} {"Aug val":>14} {"Baseline test":>15} {"Aug test":>14}')
print('-' * 82)
print(f'{"PR-AUC":<22} {BASELINE["val"]["pr_auc"]:>14.4f} {val_metrics["pr_auc"]:>14.4f} '
      f'{BASELINE["test"]["pr_auc"]:>15.4f} {test_metrics["pr_auc"]:>14.4f}')
print(f'{"ROC-AUC":<22} {BASELINE["val"]["roc_auc"]:>14.4f} {val_metrics["roc_auc"]:>14.4f} '
      f'{BASELINE["test"]["roc_auc"]:>15.4f} {test_metrics["roc_auc"]:>14.4f}')
print(f'{"Precision":<22} {BASELINE["val"]["precision"]:>14.4f} {val_metrics["precision"]:>14.4f} '
      f'{BASELINE["test"]["precision"]:>15.4f} {test_metrics["precision"]:>14.4f}')
print(f'{"Recall":<22} {BASELINE["val"]["recall"]:>14.4f} {val_metrics["recall"]:>14.4f} '
      f'{BASELINE["test"]["recall"]:>15.4f} {test_metrics["recall"]:>14.4f}')
print(f'{"F1":<22} {BASELINE["val"]["f1"]:>14.4f} {val_metrics["f1"]:>14.4f} '
      f'{BASELINE["test"]["f1"]:>15.4f} {test_metrics["f1"]:>14.4f}')

Results comparison (threshold = 0.5):
Metric                   Baseline val        Aug val   Baseline test       Aug test
----------------------------------------------------------------------------------
PR-AUC                         0.5374         0.5388          0.5309         0.5084
ROC-AUC                        0.6687         0.6791          0.6744         0.6550
Precision                      0.4580         0.4786          0.4992         0.4726
Recall                         0.5473         0.5928          0.5516         0.5516
F1                             0.4987         0.5296          0.5241         0.5090


In [10]:
# Permutation importance on context features only
# Permutes only the last 6 columns (the context features) to measure
# how much each contributes to test PR-AUC.
n_base_cols = X_base_test.shape[1]
ctx_col_start = n_base_cols

result = permutation_importance(
    model,
    X_aug_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring='average_precision',
)

print('Permutation importance for context features (test set, scoring=average_precision):')
print(f'{"Feature":<32} {"Mean":>10} {"Std":>10}')
print('-' * 55)
ctx_importances = {}
for i, name in enumerate(ctx_enc.feature_names_):
    col_idx = ctx_col_start + i
    mean_imp = result.importances_mean[col_idx]
    std_imp  = result.importances_std[col_idx]
    ctx_importances[name] = {'mean': float(mean_imp), 'std': float(std_imp)}
    print(f'{name:<32} {mean_imp:>10.4f} {std_imp:>10.4f}')

Permutation importance for context features (test set, scoring=average_precision):
Feature                                Mean        Std
-------------------------------------------------------
repo_rejection_rate                  0.0306     0.0067
agent_rejection_rate                -0.0001     0.0001
agent_repo_rejection_rate           -0.0007     0.0036
log1p_repo_pr_count                  0.0048     0.0018
log1p_agent_pr_count                 0.0023     0.0013
pr_size_vs_repo_median               0.0035     0.0077


In [11]:
# Save all exploratory artifacts
OUT_DIR = Path(DRIVE_BASE) / 'results' / 'exploratory' / 'exploratory_multimodal_lr_repo_context'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fitted RepositoryContextEncoder
enc_path = OUT_DIR / 'repo_context_encoder.joblib'
joblib.dump(ctx_enc, enc_path)
print(f'Saved: {enc_path}')

# Freshly fitted FeaturePipeline (independent of results/final/multimodal_lr/)
pipeline_path = OUT_DIR / 'feature_pipeline_multimodal_lr_repo_context.joblib'
joblib.dump(pipeline, pipeline_path)
print(f'Saved: {pipeline_path}')

# Fitted logistic regression model
model_path = OUT_DIR / 'lr_model_multimodal_lr_repo_context.joblib'
joblib.dump(model, model_path)
print(f'Saved: {model_path}')

# Val and test metrics for both baseline (hardcoded) and augmented (computed)
metrics_payload = {
    'baseline': {
        'val':  {
            'pr_auc':    BASELINE['val']['pr_auc'],
            'roc_auc':   BASELINE['val']['roc_auc'],
            'precision': BASELINE['val']['precision'],
            'recall':    BASELINE['val']['recall'],
            'f1':        BASELINE['val']['f1'],
        },
        'test': {
            'pr_auc':    BASELINE['test']['pr_auc'],
            'roc_auc':   BASELINE['test']['roc_auc'],
            'precision': BASELINE['test']['precision'],
            'recall':    BASELINE['test']['recall'],
            'f1':        BASELINE['test']['f1'],
        },
    },
    'augmented': {
        'val':  {
            'pr_auc':    val_metrics['pr_auc'],
            'roc_auc':   val_metrics['roc_auc'],
            'precision': val_metrics['precision'],
            'recall':    val_metrics['recall'],
            'f1':        val_metrics['f1'],
        },
        'test': {
            'pr_auc':    test_metrics['pr_auc'],
            'roc_auc':   test_metrics['roc_auc'],
            'precision': test_metrics['precision'],
            'recall':    test_metrics['recall'],
            'f1':        test_metrics['f1'],
        },
    },
}
metrics_path = OUT_DIR / 'metrics_multimodal_lr_repo_context.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)
print(f'Saved: {metrics_path}')

# Permutation importance for the 6 context features
perm_path = OUT_DIR / 'permutation_importance_multimodal_lr_repo_context.json'
with open(perm_path, 'w') as f:
    json.dump(ctx_importances, f, indent=2)
print(f'Saved: {perm_path}')

Saved: /content/drive/MyDrive/Thesis/AgenticPRRejection/results/exploratory/exploratory_multimodal_lr_repo_context/repo_context_encoder.joblib
Saved: /content/drive/MyDrive/Thesis/AgenticPRRejection/results/exploratory/exploratory_multimodal_lr_repo_context/feature_pipeline_multimodal_lr_repo_context.joblib
Saved: /content/drive/MyDrive/Thesis/AgenticPRRejection/results/exploratory/exploratory_multimodal_lr_repo_context/lr_model_multimodal_lr_repo_context.joblib
Saved: /content/drive/MyDrive/Thesis/AgenticPRRejection/results/exploratory/exploratory_multimodal_lr_repo_context/metrics_multimodal_lr_repo_context.json
Saved: /content/drive/MyDrive/Thesis/AgenticPRRejection/results/exploratory/exploratory_multimodal_lr_repo_context/permutation_importance_multimodal_lr_repo_context.json
